# Unit tests of MCP server called disaster-server

Unit tests for `query_disasters`

In [ ]:
%pip install pandas mcp[cli] python-dotenv openai langchain langchain-openai -q

Load the MCP server

In [1]:
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path("..") / "disasters-server" / "src"))
from disasters_server.server import query_disasters

Loaded 1900-2021: 16126 rows, 45 columns
Loaded 1970-2021: 14644 rows, 47 columns
Combined disasters data: 30770 rows, 47 columns


Unit Test baseline

In [2]:
passed = 0
failed = 0

def test(name, condition, detail=""):
    global passed, failed
    if condition:
        passed += 1
        print(f"  PASS: {name}")
    else:
        failed += 1
        print(f"  FAIL: {name} -- {detail}")

### Test: query by country


In [3]:
# Test: query by country
result = await query_disasters(country="Japan", limit=5)
data = json.loads(result)
test("query_by_country returns results", data["total"] > 0)
test("query_by_country all match Japan",
     all(d["Country"] == "Japan" for d in data["disasters"]))

  PASS: query_by_country returns results
  PASS: query_by_country all match Japan


### Test: query by year

In [4]:
# Test: query by year
result = await query_disasters(year=2010, limit=5)
data = json.loads(result)
test("query_by_year returns results", data["total"] > 0)
test("query_by_year all match 2010",
     all(d["Year"] == 2010 for d in data["disasters"]))

  PASS: query_by_year returns results
  PASS: query_by_year all match 2010


### Test: query by disaster type

In [5]:
# Test: query by disaster type
result = await query_disasters(disaster_type="Earthquake", limit=5)
data = json.loads(result)
test("query_by_type returns results", data["total"] > 0)
test("query_by_type all match Earthquake",
     all("Earthquake" in d.get("Disaster Type", "") for d in data["disasters"]))

  PASS: query_by_type returns results
  PASS: query_by_type all match Earthquake


### Test: combined filters

In [6]:
# Test: combined filters
result = await query_disasters(country="Colombia", year=2021, disaster_type="Flood", limit=5)
data = json.loads(result)
test("combined_filters returns results", data["total"] > 0)
test("combined_filters country match",
     all(d["Country"] == "Colombia" for d in data["disasters"]))
test("combined_filters year match",
     all(d["Year"] == 2021 for d in data["disasters"]))

  PASS: combined_filters returns results
  PASS: combined_filters country match
  PASS: combined_filters year match


### Test: no results

In [7]:
# Test: no results
result = await query_disasters(country="Atlantis", year=9999)
test("no_results returns message", result == "No disasters found matching the criteria.")

  PASS: no_results returns message


### Test: default limit

In [8]:
# Test: default limit
result = await query_disasters(country="China")
data = json.loads(result)
test("default_limit caps at 10", data["total"] <= 10)

  PASS: default_limit caps at 10


### Test: result structure

In [9]:
# Test: result structure
result = await query_disasters(country="Japan", year=2011, limit=1)
data = json.loads(result)
disaster = data["disasters"][0]
required_fields = ["Year", "Country", "Disaster Type", "Total Deaths"]
for field in required_fields:
    test(f"result has field '{field}'", field in disaster)

  PASS: result has field 'Year'
  PASS: result has field 'Country'
  PASS: result has field 'Disaster Type'
  PASS: result has field 'Total Deaths'


In [10]:
print(f"\n{'='*40}")
print(f"RESULTS: {passed} passed, {failed} failed, {passed+failed} total")
print(f"{'='*40}")
assert failed == 0, f"{failed} test(s) failed!"


RESULTS: 15 passed, 0 failed, 15 total
